In [ ]:
!pip install Flask

In [12]:
import numpy as np
import pandas as pd
from transformers import AutoProcessor

model_id = "microsoft/layoutlmv3-base"
processor = AutoProcessor.from_pretrained(model_id) 
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 1.23.5
Pandas: 1.5.3


In [5]:
import transformers
print(transformers.__version__)


C:\Users\HP\anaconda3\envs\layoutlmv3_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.50.3


In [7]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)


TensorFlow: 2.15.0


In [9]:
import sys
print("Python:", sys.version)

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]


In [2]:
from PIL import Image
import pandas as pd
from pathlib import Path
import numpy as np
import tensorflow as tf

# Vérifie la version et l'accès GPU
print("TensorFlow:", tf.__version__)

TensorFlow: 2.15.0


In [11]:
! pip install transformers
! pip install datasets
! pip install evaluate
! pip install seqeval
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [1]:
labels_list = ['O', 'B-COMPANY', 'B-DATE', 'B-ADDRESS', 'B-TOTAL', 'B-INVOICE_NO', 'I-INVOICE_NO',
    'B-ITEM_DESC', 'I-ITEM_DESC',
    'B-ITEM_QTY', 'I-ITEM_QTY',
    'B-ITEM_PRICE', 'I-ITEM_PRICE']
ids2labels = {k: v for k, v in enumerate(labels_list)} 
labels2ids = {v: k for k, v in enumerate(labels_list)}

In [ ]:
from difflib import SequenceMatcher

def get_word_tag(word, entities):

    word_lower = word.lower()
    

    for entity in entities:
        entity_upper = entity.upper()  # Normalisation en majuscules
        entity_value = str(entities[entity]).lower()
        
        # Logique originale inchangée
        if SequenceMatcher(None, word_lower, entity_value).ratio() >= 0.8:
            return labels2ids[f'B-{entity_upper}']
        elif entity_upper in ['ADDRESS', 'COMPANY'] and word_lower in entity_value:
            return labels2ids[f'B-{entity_upper}']
        elif entity_upper in ['DATE', 'TOTAL'] and entity_value in word_lower:
            return labels2ids[f'B-{entity_upper}']

    if 'items' in entities:
        for item in entities['items']:
            # Description
            desc = str(item.get('description', '')).lower()
            if SequenceMatcher(None, word_lower, desc).ratio() >= 0.7:
                return labels2ids['B-ITEM_DESC']
            
            # Quantité
            qty = str(item.get('quantity', '')).lower()
            if word_lower == qty:
                return labels2ids['B-ITEM_QTY']
            
            # Prix
            price = str(item.get('unit_price', '')).lower()
            if word_lower in price.replace(" ", ""):
                return labels2ids['B-ITEM_PRICE']

    return labels2ids['O']

In [5]:
def get_normalized_bbox(line, img_width, img_height):
  line = line.strip().split(',')
  x1, y1, x2, y2 = int(line[0]), int(line[1]), int(line[6]), int(line[7])
  return [x1 / img_width * 1000, y1 / img_height * 1000, x2 / img_width * 1000, y2 / img_height * 1000]

def get_word(line):
  line = line.strip().split(',')
  return ','.join(line[8:])

In [7]:
import json

def get_entities_dict(entities_file_path):
    """
    Version corrigée avec :
    - Bonne indentation
    - Gestion des valeurs None
    - Context manager pour les fichiers
    - Conversion sécurisée en string
    """
    def safe_str(value):
        """Convertit une valeur en string vide si None"""
        return str(value) if value is not None else ""
    
    with open(entities_file_path, 'r', encoding='utf-8') as entities_file:
        entities = json.load(entities_file)
    
    res_dict = {
        'COMPANY': safe_str(entities.get('company')),
        'DATE': safe_str(entities.get('date')),
        'ADDRESS': safe_str(entities.get('address')),
        'TOTAL': safe_str(entities.get('total')),
        'invoice_no': safe_str(entities.get('invoice_no')),
        'items': []
    }
    
    # Traitement des items
    for item in entities.get('items', []):
        res_dict['items'].append({
            'description': safe_str(item.get('description')),
            'quantity': safe_str(item.get('quantity')),
            'unit_price': safe_str(item.get('unit_price'))
        })
    
    return res_dict

In [9]:
from PIL import Image
import pandas as pd
from pathlib import Path
import numpy as np
print(np.__version__)
def get_image_dimensions(img_path):
  img = Image.open(img_path)
  return img.size

def create_df(data_dir):
  box_path = os.path.join(data_dir, 'box')
  img_path = os.path.join(data_dir, 'img')
  entities_path = os.path.join(data_dir, 'entities_gemini')
  print(f"Chemins utilisés :\n  📦 Box : {box_path}\n  🖼️ Image : {img_path}\n  🧠 Entities : {entities_path}")
  image_path_list = []
  bboxes_list = []
  words_list = []
  ner_tags_list = []

  for file in os.listdir(img_path):
    id = Path(file).stem

    box_file_path = os.path.join(box_path, f'{id}.txt')
    entities_file_path = os.path.join(entities_path, f'{id}.txt')

    img_width, img_height = get_image_dimensions(os.path.join(img_path, file))

    try:
      with open(box_file_path, 'r') as f:
        lines = f.readlines()
        words = [get_word(line) for line in lines]
        bboxes = [get_normalized_bbox(line, img_width, img_height) for line in lines]

      entities = get_entities_dict(entities_file_path)
      ner_tags = [get_word_tag(word, entities) for word in words]

      image_path_list.append(os.path.join(img_path, file))
      bboxes_list.append(bboxes)
      words_list.append(words)
      ner_tags_list.append(ner_tags)
    except:
      pass

  df = pd.DataFrame({
      'image_path': image_path_list,
      'bboxes': bboxes_list,
      'words': words_list,
      'ner_tags': ner_tags_list
  })

  return df

1.23.5


In [29]:
train_df = create_df(r'C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\train')
test_df = create_df(r'C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\test')
len(train_df), len(test_df)

Chemins utilisés :
  📦 Box : C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\train\box
  🖼️ Image : C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\train\img
  🧠 Entities : C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\train\entities_gemini
Chemins utilisés :
  📦 Box : C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\test\box
  🖼️ Image : C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\test\img
  🧠 Entities : C:\Users\HP\.cache\kagglehub\datasets\urbikn\sroie-datasetv2\versions\4\SROIE2019\test\entities_gemini


(626, 345)

In [ ]:
def create_stats_df(df):
   
    stats = {
        #
        'B-COMPANY': [],
        'B-DATE': [],
        'B-ADDRESS': [],
        'B-TOTAL': [],
        
        # 
        'B-INVOICE_NO': [],
        'I-INVOICE_NO': [],
        
        # 
        'B-ITEM_DESC': [],
        'I-ITEM_DESC': [],
        'B-ITEM_QTY': [],
        'I-ITEM_QTY': [],
        'B-ITEM_PRICE': [],
        'I-ITEM_PRICE': [],
        
        # 
        'not_other': [],
        'total_words': []
    }

    for _, row in df.iterrows():
        tags = row['ner_tags']
        words = row['words']
        
        # 
        counts = {label: 0 for label in stats}
        counts['total_words'] = len(words)
        
        for tag_id in tags:
            label = ids2labels[tag_id]
            if label in counts:
                counts[label] += 1
            if tag_id != labels2ids['O']:
                counts['not_other'] += 1
        
        # 
        for label in stats:
            stats[label].append(counts[label])
    
    return pd.DataFrame(stats)

# 
stats_df = create_stats_df(train_df)

#
selected_cols = [
    'B-COMPANY', 'B-DATE', 'B-ADDRESS', 'B-TOTAL',
    'B-INVOICE_NO', 'B-ITEM_DESC', 'B-ITEM_QTY',
    'B-ITEM_PRICE', 'not_other', 'total_words'
]
stats_df[selected_cols].describe()

,B-COMPANY,B-DATE,B-ADDRESS,B-TOTAL,B-INVOICE_NO,B-ITEM_DESC,B-ITEM_QTY,B-ITEM_PRICE,not_other,total_words
count,626.000000,626.000000,626.000000,626.000000,626.000000,626.000000,626.000000,626.000000,626.000000,626.000000
mean,2.202875,1.153355,5.311502,3.049521,0.399361,2.691693,0.774760,3.049521,18.632588,53.715655
std,2.397744,0.390442,3.726287,1.697747,0.493412,2.417105,1.641093,3.853199,9.508354,17.907290
min,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,7.000000,18.000000
25%,1.000000,1.000000,3.000000,2.000000,0.000000,1.000000,0.000000,0.000000,13.000000,42.000000
50%,1.000000,1.000000,4.000000,3.000000,0.000000,2.000000,0.000000,2.000000,16.000000,50.000000
75%,2.000000,1.000000,7.000000,4.000000,1.000000,4.000000,1.000000,4.000000,21.000000,63.000000
max,17.000000,3.000000,23.000000,11.000000,2.000000,17.000000,13.000000,42.000000,76.000000,153.000000


In [ ]:
def recompute_ner_tags(row):
    ner_tags = row['ner_tags']
    bboxes = row['bboxes']
    
    new_ner_tags = [labels2ids['O'] for _ in ner_tags]
    
    # 
    tag_strategies = {
        'B-COMPANY': {
            'condition': lambda x: True,  # Pas de condition particulière
            'selector': lambda idx_list, bboxes: max(idx_list, key=lambda x: (bboxes[x][2] - bboxes[x][0]) * (bboxes[x][3] - bboxes[x][1])),  # Plus grande bbox
            'count': 'single' 
        },
        'B-ADDRESS': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: max(idx_list, key=lambda x: (bboxes[x][2] - bboxes[x][0]) * (bboxes[x][3] - bboxes[x][1])),
            'count': 'single'
        },
        'B-DATE': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: min(idx_list, key=lambda x: (bboxes[x][1] + bboxes[x][3])/2),  # Bbox la plus haute
            'count': 'single'
        },
        'B-TOTAL': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: max(idx_list, key=lambda x: (bboxes[x][1] + bboxes[x][3])/2),  # Bbox la plus basse
            'count': 'single'
        },
        'B-INVOICE_NO': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: min(idx_list, key=lambda x: (bboxes[x][0] + bboxes[x][2])/2),  # Bbox la plus à gauche
            'count': 'single'
        },
        # 
        'I-ITEM_DESC': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: idx_list,  # Garde tous
            'count': 'multiple'
        },
        'I-ITEM_QTY': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: idx_list,
            'count': 'multiple'
        },
        'I-ITEM_PRICE': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: idx_list,
            'count': 'multiple'
        },
        'I-INVOICE_NO': {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: idx_list,
            'count': 'multiple'
        }
    }
    
  
    for tag in labels_list:
        if tag == 'O' or tag not in labels2ids:
            continue
            
        tag_id = labels2ids[tag]
        tag_idx = [i for i, t in enumerate(ner_tags) if t == tag_id]
        
        if not tag_idx:
            continue
            
        strategy = tag_strategies.get(tag, {
            'condition': lambda x: True,
            'selector': lambda idx_list, bboxes: idx_list[0],  # Par défaut: premier élément
            'count': 'single'
        })
        
        selected_idx = strategy['selector'](tag_idx, bboxes)
        
        if strategy['count'] == 'single':
            if isinstance(selected_idx, list):
                selected_idx = selected_idx[0] if selected_idx else None
            if selected_idx is not None:
                new_ner_tags[selected_idx] = tag_id
        else:  # multiple
            for idx in selected_idx:
                new_ner_tags[idx] = tag_id
    
    return new_ner_tags

In [35]:
train_df['ner_tags'] = train_df.apply(recompute_ner_tags, axis=1)
test_df['ner_tags'] = test_df.apply(recompute_ner_tags, axis=1)

In [37]:
train_stats_df = create_stats_df(train_df)
train_stats_df.describe()

,B-COMPANY,B-DATE,B-ADDRESS,B-TOTAL,B-INVOICE_NO,I-INVOICE_NO,B-ITEM_DESC,I-ITEM_DESC,B-ITEM_QTY,I-ITEM_QTY,B-ITEM_PRICE,I-ITEM_PRICE,not_other,total_words
count,626.000000,626.000000,626.000000,626.0,626.000000,626.0,626.000000,626.0,626.00000,626.0,626.000000,626.0,626.000000,626.000000
mean,0.998403,0.995208,0.974441,1.0,0.397764,0.0,0.926518,0.0,0.28115,0.0,0.694888,0.0,6.268371,53.715655
std,0.039968,0.069116,0.157942,0.0,0.489828,0.0,0.261135,0.0,0.44992,0.0,0.460823,0.0,0.921663,17.907290
min,0.000000,0.000000,0.000000,1.0,0.000000,0.0,0.000000,0.0,0.00000,0.0,0.000000,0.0,4.000000,18.000000
25%,1.000000,1.000000,1.000000,1.0,0.000000,0.0,1.000000,0.0,0.00000,0.0,0.000000,0.0,6.000000,42.000000
50%,1.000000,1.000000,1.000000,1.0,0.000000,0.0,1.000000,0.0,0.00000,0.0,1.000000,0.0,6.000000,50.000000
75%,1.000000,1.000000,1.000000,1.0,1.000000,0.0,1.000000,0.0,1.00000,0.0,1.000000,0.0,7.000000,63.000000
max,1.000000,1.000000,1.000000,1.0,1.000000,0.0,1.000000,0.0,1.00000,0.0,1.000000,0.0,8.000000,153.000000


In [39]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset, test_dataset

(Dataset({
     features: ['image_path', 'bboxes', 'words', 'ner_tags'],
     num_rows: 626
 }),
 Dataset({
     features: ['image_path', 'bboxes', 'words', 'ner_tags'],
     num_rows: 345
 }))

In [41]:
!pip install google-generativeai pillow python-dotenv

  Using cached google_generativeai-0.8.4-py3-none-any.whl.metadata (4.2 kB)
  Using cached python_dotenv-1.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.24.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached google_api_python_client-2.166.0-py2.py3-none-any.whl.metadata (6.6 kB)
  Using cached pydantic-2.11.3-py3-none-any.whl.metadata (65 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.69.2-py3-none-any.whl.metadata (9.3 kB)
  Using cached httplib2-0.22.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached uritemplate-4.1.1-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached grpcio_status-1

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.4 which is incompatible.


In [1]:
import pyarrow
print(pyarrow.__version__)

19.0.1


In [ ]:
# Model Configuration
MODEL_CONFIG = {
  "temperature": 0.2,
  "top_p": 1,
  "top_k": 32,
  "max_output_tokens": 4096,
}

## 
safety_settings = [
  {
    "category": "HARM_CATEGORY_HARASSMENT",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
  {
    "category": "HARM_CATEGORY_HATE_SPEECH",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
  {
    "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
  {
    "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  }
]

In [ ]:
def get_model_path(config_path="config.json"):
   
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Fichier de configuration introuvable: {config_path}")
    
  
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    if "model_path" not in config:
        raise ValueError("Le fichier config doit contenir une clé 'model_path'")
    
    model_path = config["model_path"]
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Le chemin du modèle spécifié n'existe pas: {model_path}")
    
    return model_path

In [ ]:
import os
from transformers import AutoModelForTokenClassification, AutoProcessor
import os
from pathlib import Path  
os.environ["TOKENIZERS_PARALLELISM"] = "false"
bases_folder = Path('C:/Users/HP/Desktop/sequencedemodel')
config_path = bases_folder / "modelsfinals" / "config.json.txt"
path = get_model_path(config_path=config_path)
print(path) 
print(path)

C:\Users\HP\anaconda3\envs\layoutlmv3_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



C:\Users\HP\Downloads\modelfinal\modelfinal
C:\Users\HP\Downloads\modelfinal\modelfinal


In [ ]:
from transformers import AutoTokenizer
path_to_model = fr"{path}" 
tokenizer = AutoTokenizer.from_pretrained(path_to_model,local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(path_to_model)

In [6]:
import torch
from collections import Counter

class ReceiptReader:
  def __init__(self, path_to_model=fr"{path}"):
    self.model = AutoModelForTokenClassification.from_pretrained(path_to_model)
    self.model.eval()
    self.processor = AutoProcessor.from_pretrained(path_to_model)      

  def __call__(self, image):
    encodings = self.__get_encodings(image)
    words = self.__get_words(encodings)
    bboxes = encodings.bbox[0]
    logits = self.model(**encodings).logits
    predictions = torch.argmax(logits, dim=2)
    labeled_tokens = [self.model.config.id2label[t.item()] for t in predictions[0]]
    response_dict = self.__merge_tokens(words, bboxes, labeled_tokens)
    response_dict["bboxes"] = [self.__unnormalize_bbox(bbox, image) for bbox in response_dict["bboxes"]]
    return response_dict
  
  def __get_encodings(self, image):
    return self.processor(image, return_tensors="pt")
  
  def __get_words(self, encodings):
    words = [self.processor.tokenizer.decode(input_id) for input_id in encodings.input_ids[0]]
    return words
  
  def __merge_tokens(self, words, bboxes, labels):
    new_words = []
    new_bboxes = []
    new_labels = []
    i = 0
    while i < len(words):
        token, bbox, label = words[i], bboxes[i], labels[i]
        j = i + 1
        while j < len(words) and self.__is_same_bbox(bbox, bboxes[j]):
            token += words[j]
            j += 1
        counter = Counter([labels[k] for k in range(i, j)])
        sorted_labels = sorted(counter, key=counter.get, reverse=True)
        if sorted_labels[0] == "O" and len(sorted_labels) > 1:
          label = sorted_labels[1]
        else:
          label = sorted_labels[0]
        new_words.append(token)
        new_bboxes.append(bbox)
        new_labels.append(label)
        i = j
    return {
        "words": new_words,
        "bboxes": new_bboxes,
        "labels": new_labels
    }
  
  def __is_same_bbox(self, bbox1, bbox2):
    for i in range(4):
        if abs(bbox1[i] - bbox2[i]) > 3:
            return False
    return True
  
  def __unnormalize_bbox(self, bbox, image):
    width, height = image.size
    return [bbox[0] * width / 1000, bbox[1] * height / 1000, bbox[2] * width / 1000, bbox[3] * height / 1000]

In [9]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [13]:
class ReceiptInformationExtractor:
    def __init__(self, path_to_model=fr"{path}"):
        self.receipt_reader = ReceiptReader(path_to_model)
    
    def __call__(self, image):
        receipt_data = self.receipt_reader(image)
        response_dict = {
            "company": "",
            "date": "",
            "address": "",
            "total": "",
            "invoice_no": "",
            "items": []
        }
        
        # company akbir bbox
        max_bbox = 0
        for word, bbox, label in zip(receipt_data['words'], receipt_data["bboxes"], receipt_data["labels"]):
            if label == "B-COMPANY":
                bbox_size = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
                if bbox_size > max_bbox:
                    response_dict["company"] = word.strip()
                    max_bbox = bbox_size
        
        # address akbir bbox
        max_bbox = 0
        for word, bbox, label in zip(receipt_data['words'], receipt_data["bboxes"], receipt_data["labels"]):
            if label == "B-ADDRESS":
                bbox_size = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
                if bbox_size > max_bbox:
                    response_dict["address"] = word.strip()
                    max_bbox = bbox_size
        
        # date
        min_y = float("inf")
        for word, bbox, label in zip(receipt_data['words'], receipt_data["bboxes"], receipt_data["labels"]):
            if label == "B-DATE" and bbox[1] < min_y:
                response_dict["date"] = word.strip()
                min_y = bbox[1]
        
        # total
        max_y = 0
        for word, bbox, label in zip(receipt_data['words'], receipt_data["bboxes"], receipt_data["labels"]):
            if label == "B-TOTAL" and bbox[3] > max_y:
                response_dict["total"] = word.strip()
                max_y = bbox[3]
        
        # invoice num
        invoice_parts = []
        invoice_positions = []
        for word, bbox, label in zip(receipt_data['words'], receipt_data["bboxes"], receipt_data["labels"]):
            if label == "B-INVOICE_NO" or label == "I-INVOICE_NO":
                invoice_parts.append((word.strip(), bbox))
                invoice_positions.append(bbox[1])  # y-coordinate for sorting
        
        # Sort invoice parts by vertical position
        if invoice_parts:
            sorted_invoice = [part for _, part in sorted(zip(invoice_positions, invoice_parts), key=lambda x: x[0])]
            response_dict["invoice_no"] = " ".join([part[0] for part in sorted_invoice])
        
        # Process items
        current_item = None
        items = []
        
        # all words belonging to items and their positions
        item_words = []
        for word, bbox, label in zip(receipt_data['words'], receipt_data["bboxes"], receipt_data["labels"]):
            if any(label.startswith(prefix) for prefix in ["B-ITEM_DESC", "I-ITEM_DESC", 
                                                         "B-ITEM_QTY", "I-ITEM_QTY", 
                                                         "B-ITEM_PRICE", "I-ITEM_PRICE"]):
                item_words.append((word.strip(), bbox, label))
        
        # Sort by y-coordinate to items
        item_words.sort(key=lambda x: x[1][1])
        
        # Process sorted item words to build items list
        for word, bbox, label in item_words:
            if label == "B-ITEM_DESC":
                if current_item:
                    items.append(current_item)
                current_item = {"description": word, "quantity": "", "unit_price": ""}
            elif label == "I-ITEM_DESC" and current_item:
                current_item["description"] += " " + word
            elif label == "B-ITEM_QTY":
                if not current_item:
                    current_item = {"description": "", "quantity": word, "unit_price": ""}
                else:
                    current_item["quantity"] = word
            elif label == "I-ITEM_QTY" and current_item:
                current_item["quantity"] += " " + word
            elif label == "B-ITEM_PRICE":
                if not current_item:
                    current_item = {"description": "", "quantity": "", "unit_price": word}
                else:
                    current_item["unit_price"] = word
            elif label == "I-ITEM_PRICE" and current_item:
                current_item["unit_price"] += " " + word
        
        # last item 
        if current_item:
            items.append(current_item)
        
        response_dict["items"] = items
        
        return response_dict

In [21]:
import socket
host = socket.gethostbyname(socket.gethostname())  # Ex: '192.168.1.157'
print(host);

192.168.1.158


In [ ]:
from flask import Flask, request, jsonify
import PIL.Image

app = Flask(__name__)

@app.route('/process-invoice', methods=['POST'])
def process_invoice():
    try: 
        data = request.json
        if not data or "image_path" not in data:
            return jsonify({"error": "Missing 'image_path' in request"}), 400
        
        image_path = data["image_path"]
        print(image_path); 
        image = PIL.Image.open(image_path).convert("RGB")
        receipt_info_extractor = ReceiptInformationExtractor()
        result = receipt_info_extractor(image)
        return jsonify(result)

    except FileNotFoundError as e:
        return jsonify({"error": f"Image not found: {str(e)}"}), 404
    except PIL.UnidentifiedImageError:
        return jsonify({"error": "Invalid image format"}), 400
    except Exception as e:
        return jsonify({"error": f"Server error: {str(e)}"}), 500


In [ ]:
if __name__ == '__main__':
    app.run(host=host, port=5002, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://192.168.1.158:5002
Press CTRL+C to quit


C:/Users/HP/back_end_final/uploads2/1746207682106-processed_invoice_1746207681815.jpg


192.168.1.158 - - [02/May/2025 19:41:24] "POST /process-invoice HTTP/1.1" 200 -


C:/Users/HP/back_end_final/uploads2/1746207780062-processed_invoice_1746207779882.jpg


...

192.168.1.158 - - [02/May/2025 20:14:45] "POST /process-invoice HTTP/1.1" 200 -


C:/Users/HP/back_end_final/uploads2/1746209885812-processed_invoice_1746209884761.jpg


192.168.1.158 - - [02/May/2025 20:18:07] "POST /process-invoice HTTP/1.1" 200 -


C:/Users/HP/back_end_final/uploads2/1746209996196-processed_invoice_1746209995437.jpg


192.168.1.158 - - [02/May/2025 20:19:57] "POST /process-invoice HTTP/1.1" 200 -
